![SNOTEL](Images/NWM.png)

# Retrieve and Analyze National Water Model Snow Data for a Watershed of Interest
Authors: Irene Garousi-Nejad (igarousi@cuahsi.org)
Last updated: July 10, 2025

**Introduction**: This notebook is divided into three main sections, each demonstrating how to retrieve snow water equivalent (SWE) data from different sources. This workflow is organized into six main sections. The first section ensures that the necessary Python packages and tools are installed and ready for analysis. The second section defines key parameters such as the watershed boundary and the locations of observation sites. The third section, Retrieve Observed Snow Data, focuses on acquiring observed SWE data from SNOTEL sites located within the Tuolumne River watershed. In the fourth section, Retrieve Modeled Snow Data, we access retrospective SWE data from National Water Model (NWM) version 3. The fifth section performs a direct comparison between modeled and observed SWE at individual SNOTEL sites. Finally, the sixth section evaluates how modeled and observed SWE compare when aggregated across the entire Tuolumne River watershed.

### 1. Prepare the Python Environment

Ensure that the `HydroLearnNWMEnv` conda environment is selected as your Jupyter kernel. This environment should already be created if you followed the instructions under section "Creating your HydroLearnEnv Virtual Environment" in the `getting_started.md` file.

In [ ]:
# Confirm the kernel is running from HydroLearnEnv
!which python

Import the libraries needed to run this notebook:

In [ ]:
import os
import sys
prefix = os.environ['CONDA_PREFIX']
os.environ['GDAL_DATA'] = os.path.join(prefix, 'share', 'gdal')
os.environ['PROJ_LIB'] = os.path.join(prefix, 'share', 'proj')

import dask
import numpy
import glob
import pyproj
import pandas as pd
import xarray as xr
import time
import fsspec
import rioxarray
import hvplot.pandas
from supporting_scripts import nwm_utils
import hvplot.pandas
import holoviews as hv
import hvplot.xarray
hv.extension('bokeh')
import geopandas as gpd
import matplotlib.pyplot as plt
from dask.distributed import Client
import panel as pn
pn.extension()

import warnings
warnings.filterwarnings("ignore")

We'll use dask to parallelize our code. To manage parallel computation and visualize progress of long-running tasks, we initialize a Dask “cluster,” which defines how many workers are used and how much computing power each worker has. 

In this setup, we create a Dask client with `Client(n_workers=6, threads_per_worker=1, memory_limit='2GB')`, which launches a cluster with 6 workers. Each worker uses a single thread, typically mapped to one CPU core, allowing for efficient parallel processing across 6 cores. Each worker also has a memory limit of 2 GB, for a total of up to 12 GB across the cluster.


In [ ]:
# use a try accept loop so we only instantiate the client
# if it doesn't already exist.
try:
    print('Dashboard link:', client.dashboard_link)
except:    
    # The client should be customized to your workstation resources.
    client = Client(n_workers=6, threads_per_worker=1, memory_limit='2GB') 
    print('Dashboard link:', client.dashboard_link)
print(client)

### 2. Set Inputs

In [ ]:
# Path to the watershed shapefile
watershed = "./files/TolumneRiver_18040009.shp"
basin = "./files/DonPedroDam_Upstream_Basin.shp"

# Path to NWM snow data
conus_bucket_url = 's3://noaa-nwm-retrospective-3-0-pds/CONUS/zarr/ldasout.zarr'

# Start and end times of a water year (note that this code currently works for one water year)
StartDate = '2018-10-01'
EndDate = '2019-10-01'

# Path to save results (obs and mod stands for observation and modeled, respectively)
OBS_OutputFolder = './obs_outputs' 
MOD_OutputFolder = './mod_outputs'

### 3. Retrieve Observed Snow Data 

This code reads a GeoJSON file of all snow observation stations and filters them to include only stations in California with available CSV data. It also loads the watershed boundary shapefile (`TolumneRiver_18040009.shp`), converts it to the appropriate coordinate system, and merges all watershed polygons into a single MultiPolygon. Finally, it counts how many observation sites fall within this combined watershed boundary, providing the number of sites located inside the study area. 

**Heads up**: You might need to run the next cell twice. Sometimes, the spatial filtering does not return any results the first time due to how geometries are loaded or processed in the background.

In [ ]:
# Create geodataframe of all stations
all_stations_gdf = gpd.read_file('https://raw.githubusercontent.com/egagli/snotel_ccss_stations/main/all_stations.geojson').set_index('code')
all_stations_gdf = all_stations_gdf[all_stations_gdf['csvData']==True]
filtered_all_stations_gdf = all_stations_gdf[all_stations_gdf.state.str.contains('California')]  

# Extract the bounding box coordinates of a watershed
watershed_gdf = gpd.read_file(os.path.join(os.getcwd(), watershed)).to_crs(epsg=4326)

# Combine all polygons into a single MultiPolygon
watershed_union = watershed_gdf.geometry.unary_union

# Use the polygon geometry to select snotel sites that are within the domain
gdf_in_bbox = filtered_all_stations_gdf[filtered_all_stations_gdf.geometry.within(watershed_union)]
gdf_in_bbox.reset_index(inplace=True)
print(f'There are {len(gdf_in_bbox)} sites from', ', '.join(set(gdf_in_bbox.network)), 'network(s) within the watershed.')


Plot these sites on a map. Then, hover over the pins to see the site names.

In [ ]:
m = nwm_utils.plot_sites_within_domain(gdf_in_bbox, watershed_gdf, zoom_start=9)
m

Check the start and end date of available data for these sites.

In [ ]:
for i in gdf_in_bbox.index:
    print('Site ', gdf_in_bbox.iloc[i].code, ":", gdf_in_bbox.iloc[i].beginDate, "-", gdf_in_bbox.iloc[i].endDate)

The following uses the `nwm_utils.py` script to download observed data for the sites within the domain. Since all the sites are from the (California Cooperative Snow Survey) CCSS network, we use the `getCCSSData` function from the module to get data. 

<div style="background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
<h4>📖 Did you know?</h4>
<p>The California Cooperative Snow Surveys (CCSS) program, managed primarily by the California Department of Water Resources (DWR), monitors snowpack across key California watersheds to estimate annual water supply. Most CCSS sites are manual snow courses, where surveyors physically measure snow depth and snow water equivalent (SWE) several times a year, though some sites have been upgraded with automated sensors. In contrast, SNOTEL sites, managed by the USDA Natural Resources Conservation Service (NRCS), are fully automated and collect data continuously across the western United States, including California. While CCSS primarily supports water supply forecasting, SNOTEL supports both water supply forecasting and climate monitoring. </p>
</div>

In [ ]:
# Create a folder to save results
isExist = os.path.exists(OBS_OutputFolder)
if isExist == True:
    exit
else:
    os.mkdir(OBS_OutputFolder)

In [ ]:
for i in gdf_in_bbox.index:
    nwm_utils.getCCSSData(gdf_in_bbox.name[i], gdf_in_bbox.code[i], StartDate, EndDate, OBS_OutputFolder)

### 4. Retrieve Modeled Snow Data

NOAA shares inputs and outputs to the National Water Model retrospective simulations version 3 at <a href="https://noaa-nwm-retrospective-3-0-pds.s3.amazonaws.com/index.html" style="color: blue; background-color: snow;">https://noaa-nwm-retrospective-3-0-pds.s3.amazonaws.com/index.html</a>. The following code uses `fsspec` and `xarray` Python libraries to load the Zarr metadata of snow outputs (**ldasout.zarr**) into memory. Once the code is executed, you can see the wall time, which includes time spent waiting for I/O operations, such as reading data from a remote server. In our case, it took about 12 seconds to load the metadata into memory. Set up `Dask`, a parallel computing library, to enable performing operations on large datasets that don't fit into memory by breaking them into smaller, manageable pieces called chunks.

In [ ]:
%%time 
ds = xr.open_zarr(store=conus_bucket_url, 
                  consolidated=True,
                  storage_options={"anon": True})

The following code retrieves NWM data for each SNOTEL site within our watershed. For each site, it first converts latitude and longitude of the site to the projected coordinates used by the NWM. Then, it extracts the NWM data SWE for the site and the period of interest, saving the result as a DataFrame. Since NWM timestamps are in UTC, the DataFrame is converted to the local time zone to match SNOTEL observations for later comparison purposes. To fairly compare with SNOTEL, which reports SWE once daily at the start of the local day, the data is grouped by date, and the earliest record of each day is selected. Finally, the processed data is saved as a CSV file for each site.

In [ ]:
# Create a folder to save results
isExist = os.path.exists(MOD_OutputFolder)
if isExist == True:
    exit
else:
    os.mkdir(MOD_OutputFolder)

In [ ]:
# Retrieve data for the location of snotel sites
input_crs = 'EPSG:4269'
output_crs = pyproj.CRS(ds.crs.esri_pe_string) 

for i in range(0, gdf_in_bbox.shape[0]):
    site_name = gdf_in_bbox.iloc[i]["name"]
    print(f'[{i+1}/{len(gdf_in_bbox)}] Retrieving data for site: {site_name}')
    snotel_y, snotel_x = nwm_utils.convert_latlon_to_yx(gdf_in_bbox.iloc[i].latitude, 
                                                      gdf_in_bbox.iloc[i].longitude, 
                                                        input_crs, ds, output_crs)
    
    dl_start_time = time.time()
    ds_subset = ds[['SNEQV']].sel(y=snotel_y, x=snotel_x, method='nearest'
                                 ).sel(time=slice(StartDate, EndDate)).compute()
    dl_elapsed = time.time() - dl_start_time
    print(f'✅ Retrieved data for {site_name} in {dl_elapsed:.2f} seconds\n')
    
    df = ds_subset.to_dataframe()
    df=df.drop(columns=['x', 'y'])
    df.reset_index(inplace=True)
    df["time"] = pd.to_datetime(df["time"])
    df.rename(columns={df.columns[0]:'Date', df.columns[1]:'NWM_SWE_meters'}, inplace=True)
    df.iloc[:, 1:] = df.iloc[:, 1:].apply(lambda x: pd.to_numeric(x)/1000)  # convert mm to m   

    # convert utc to local time zone
    df_local = nwm_utils.convert_utc_to_local(gdf_in_bbox.iloc[i].state, df)   
    
    # groupby the data and select the first item from each group 
    df_local.index = pd.to_datetime(df_local['Date_Local'])
    df_local = df_local.groupby(pd.Grouper(freq='D')).first()

    # save
    df_local.to_csv(f'./{MOD_OutputFolder}/df_{gdf_in_bbox.iloc[i].code}_{gdf_in_bbox.iloc[i].state}_NWM.csv', index=False)

### 5. Compare modeled SWE with observed at a site

You should already have the CSV data for both observed and modeled flows in your working environment. The section below reads data and then for a selected gage compares model performance using various metrics.

In [ ]:
obs = glob.glob(os.path.join('obs_outputs', '*.csv'))
mod = glob.glob(os.path.join('mod_outputs', '*.csv'))

# call the function from the nwm_utils library
combined_df = nwm_utils.combine(obs, mod, StartDate, EndDate)

In [ ]:
combined_df.columns

Review the sites within the watershed from the interactive map above and click on the markers to view the site name and code. Once you’ve identified the site of interest, enter its site code in the next code cell.

In [ ]:
# choose a gage of interest within the watershed
my_site_code = 'HRS'

In [ ]:
gdf_in_bbox[gdf_in_bbox['code']=='HRS']

Plot both datasets.

In [ ]:
nwm_utils.comparison_plots(combined_df, f'CCSS_{my_site_code}_swe_m', f'NWM_{my_site_code}_swe_m')

Let's customize the scatter plot by allowing you to highlight specific months with a distinct color. The selected months will appear in one color, while all other months will appear in a different color. This customization helps reveal whether there are **seasonal patterns** in the relationship between observed and modeled SWE. In fact, it allows us to see if the model performs differently during key periods, such as the <span style="color: teal;">**accumulation**</span> phase or the <span style="color: tomato;">**melt**</span> period. Identifying these patterns is important for diagnosing the model’s strengths and limitations during different parts of the snow season, which is critical for understanding how well the model represents key processes driving snowpack evolution.

You can change the list of months to highlight (for example, 10 for October or 1 for January) to see how highlighting different parts of the year changes the appearance and interpretation of the scatter plot.

In [ ]:
combined_df['month'] = combined_df.index.month

plot = nwm_utils.plot_custom_scatter(combined_df, my_site_code, highlight_months=[10, 11, 12, 1])
plot

Compute statistics

In [ ]:
nwm_utils.compute_stats(combined_df, f'CCSS_{my_site_code}_swe_m', f'NWM_{my_site_code}_swe_m')

<div style="background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
<h4>🧠 Reflect</h4>
<p>You now have several performance metrics: Bias, Pearson Correlation, Spearman Correlation, NSE, and KGE. If you had to pick just one metric to summarize model performance, which would you choose—and why? As you review the results, compare the peak flow amounts and the timing of snowmelt onset. Do you see any significant differences? Which dataset indicates an earlier melt?</p>
</div>

Pearson and Spearman correlations are both close to 1, suggesting a strong relationship between observed and modeled SWE. As shown on the timeseries plot, this strong correlation alone does not indicate a "good" model. For example, it does not guarantee accurate timing of key events, such as peak SWE or melt onset. Let's compare these as well. The following code uses `report_max_dates_and_values` function to identify the peak SWE value and the date it occurs for both the observed (CCSS) and modeled (NWM) datasets. 

In [ ]:
summary_table = nwm_utils.report_max_dates_and_values(combined_df, f'CCSS_{my_site_code}_swe_m', f'NWM_{my_site_code}_swe_m')
summary_table

<div style="background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
  <h4>🧠 Reflect</h4>
  <p>
    What is the modeled SWE on the date when the observed SWE reaches its peak?
    Use the code snippet below to find the answer.
  </p>
  <pre><code class="language-python">
  
    # Find date of the peak SWE from observed data
    date_obs_max = combined_df['CCSS_HRS_swe_m'].idxmax()

    # Get corresponding value of modeled data on that date
    value_mod_at_max_obs = combined_df.loc[date_obs_max, 'NWM_HRS_swe_m']
  </code></pre>
</div>


Compare the average melt rate over the full melt period. 

The following function computes the melt period length by identifying the first date after the peak SWE when SWE drops to zero and remains at zero for at least (`min_zero_days`) consecutive days. This is used to define the end of the melt period. Finally, the function calculates the average melt rate, which represents the rate at which snow disappeared, expressed in meters per day, over the full melt period.

In [ ]:
observed_melt_period = nwm_utils.compute_melt_period(combined_df[f'CCSS_{my_site_code}_swe_m'])
observed_melt_period

In [ ]:
modeled_melt_period = nwm_utils.compute_melt_period(combined_df[f'NWM_{my_site_code}_swe_m'])
modeled_melt_period

### 6. Compare modeled SWE with observed over the entire watershed

In this section, we take a deeper dive into analyzing the NWM snow data across the entire Tuolumne River watershed. To do this, we need to clip the gridded dataset to the watershed boundary, enabling us to compute spatial averages over the region. To begin, we will create a single geometry for the watershed by dissolving all HUC8-level basins within it.

In [ ]:
watershed_huc8 = watershed_gdf.dissolve(by='HUC_8')

Similar to Section 4, we first ensure that the NWM data is loaded and that the domain of interest uses the same coordinate reference system (CRS) as the NWM dataset. This is required when clipping the dataset using `rioxarray`'s clip method later.

In [ ]:
%time ds = ds = xr.open_zarr(store=conus_bucket_url, consolidated=True, storage_options={"anon": True})
ds.rio.set_crs(ds.crs.attrs['esri_pe_string'])
ds.rio.write_crs(inplace=True);
watershed_huc8_prj = watershed_huc8.to_crs(ds.rio.crs)

Let's clip the gridded data to the extent of this watershed. This can be done using `rioxarray`'s "clip" method.

In [ ]:
%%time
ds_clip = ds.rio.clip(
         watershed_huc8_prj.geometry.values,
         watershed_huc8_prj.crs,
         all_touched=True,   # select all grid cells that touch the vector boundary
         drop=True,          # drop anything that is outside the clipped region
         invert=False,
         from_disk=True).sel(time=slice(StartDate, EndDate))

Preview the data at a specific point in time by defining an **exact date and time** (e.g., '2019-02-01T00:00') or an **index** (e.g., 1000, referring to the 1000th time step in the data array). We will use `hvplot` and `Holoviews` Python packages to create a more informative plot that displays both the gridded data and the vector data. One important note: for `hvplot` to visualize gridded data correctly on a map, all data must be in a **geographic coordinate system** (e.g., EPSG:4326). Note that it may take a while for the following task (map visualization) to be completed.

In [ ]:
# Explore the time values
ds_clip.time.values

In [ ]:
# Explore the variable of interest. We will be working with SNEQV, which is SWE. But feel free to explore other varibales. Note that for some of these varibales, the dimension is different and the funciton may not work for all.
ds_clip.data_vars

In [ ]:
# Define the time and variable of interest
data_var='SNEQV'
time_index='2019-01-01T00:00:00.000000000'

nwm_utils.plot_grid_vector_data(ds_clip, data_var, time_index, watershed_huc8, gdf_in_bbox)

Go ahead and play around with the time slider above to see how SWE changes over time across the watershed. 

<div style="background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
  <h4>🔍 Explore</h4>
  <p>
    Using the zoom options in the hvplot toolbar, can you comment on the possible reasons for the missing values or pixels visible on the map?
  </p>
</div>

Compute the spatial average of SWE across the entire region. First, we calculate the spatial average for both the modeled and observed datasets. Then, we create a DataFrame that combines the basin-averaged modeled data with the observed data for direct comparison.

In [ ]:
# Modeled data
ds_clip_swe_mean = ds_clip.SNEQV.mean(dim=(['x', 'y'])).sel(time=slice(StartDate, EndDate)).compute()
df_mod_swe_mean = nwm_utils.prep_nwm_swe_dataframe(ds_clip_swe_mean, gdf_in_bbox.iloc[0].state)

# Observed data
df_obs_swe_mean = nwm_utils.compute_spatial_agg_from_obs('./obs_outputs', 'mean')
df_obs_swe_mean['Date'] = pd.to_datetime(df_obs_swe_mean['Date']).dt.date
df_mod_swe_mean['Date_Local'] = pd.to_datetime(df_mod_swe_mean['Date_Local']).dt.date

# Combine
combined_df_mean = pd.merge(
    df_mod_swe_mean, df_obs_swe_mean, 
    left_on='Date_Local',  
    right_on='Date',       
    how='inner'  
)
combined_df_mean = combined_df_mean[['Date_Local', 'Snow Water Equivalent (m) Start of Day Values', 'NWM_SWE_meters']]
combined_df_mean.index = pd.to_datetime(combined_df_mean['Date_Local'])

# Plot
nwm_utils.comparison_plots(combined_df_mean, 'Snow Water Equivalent (m) Start of Day Values', 'NWM_SWE_meters')

In [ ]:
# Stats
nwm_utils.compute_stats(combined_df_mean, 'Snow Water Equivalent (m) Start of Day Values', 'NWM_SWE_meters')

<div style="background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
  <h4>🧠 Reflect</h4>
  <p>
    What could be the potential sources of discrepancies between the spatially averaged SWE from the modeled and observed datasets within this catchment?  
How could you generate more reliable or informative statistics?  
How do you think the presence of SWE values equal to zero across large areas of the gridded dataset (as seen in the map above) — particularly when this pattern is consistent across the water year — might impact the spatially averaged SWE values?  
How could this influence your interpretation?
  </p>
</div>

#### Don Pedro Reservoir Basin

Repeat the steps above, but this time focus on a smaller domain. For example, the upstream catchments of the Don Pedro Reservoir (`DonPedroDam_Upstream_Basin.shp`). This allows us to examine the impact of excluding areas where SWE is zero for the majority of days within a water year. Could this approach improve the statistical comparison between modeled and observed SWE? 

<div style="background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
  <h4>💡 Note</h4>
  <p>
    The following steps assume that the necessary code cells from the previous sections have already been executed.
  </p>
  <pre><code class="language-python">
  
    %time ds = xr.open_zarr(fsspec.get_mapper(conus_bucket_url, anon=True), consolidated=True)
    ds.rio.set_crs(ds.crs.attrs['esri_pe_string'])
    ds.rio.write_crs(inplace=True);

  </code></pre>
</div>


In [ ]:
donpedro_gdf = gpd.read_file(os.path.join(os.getcwd(), basin)).to_crs(epsg=4326)
donpedro_gdf_prj = donpedro_gdf.to_crs(ds.rio.crs)
ds_clip_donpedro = ds.rio.clip(
         donpedro_gdf_prj.geometry.values,
         donpedro_gdf_prj.crs,
         all_touched=True,   
         drop=True,         
         invert=False,
         from_disk=True).sel(time=slice(StartDate, EndDate))

In [ ]:
# Define the time and variable of interest
data_var='SNEQV'
time_index='2019-01-01T00:00:00.000000000'

nwm_utils.plot_grid_vector_data(ds_clip_donpedro, data_var, time_index, donpedro_gdf, gdf_in_bbox)

In [ ]:
# Modeled data
ds_clip_swe_mean = ds_clip_donpedro.SNEQV.mean(dim=(['x', 'y'])).sel(time=slice(StartDate, EndDate)).compute()
df_mod_swe_mean = nwm_utils.prep_nwm_swe_dataframe(ds_clip_swe_mean, gdf_in_bbox.iloc[0].state)

# Observed data
df_obs_swe_mean = nwm_utils.compute_spatial_agg_from_obs('./obs_outputs', 'mean')
df_obs_swe_mean['Date'] = pd.to_datetime(df_obs_swe_mean['Date']).dt.date
df_mod_swe_mean['Date_Local'] = pd.to_datetime(df_mod_swe_mean['Date_Local']).dt.date

# Combine with the observed data
combined_df_mean = pd.merge(
    df_mod_swe_mean, df_obs_swe_mean, 
    left_on='Date_Local',  
    right_on='Date',       
    how='inner'  
)
combined_df_mean = combined_df_mean[['Date_Local', 'Snow Water Equivalent (m) Start of Day Values', 'NWM_SWE_meters']]
combined_df_mean.index = pd.to_datetime(combined_df_mean['Date_Local'])

# Plot
nwm_utils.comparison_plots(combined_df_mean, 'Snow Water Equivalent (m) Start of Day Values', 'NWM_SWE_meters')

In [ ]:
# Stats
nwm_utils.compute_stats(combined_df_mean, 'Snow Water Equivalent (m) Start of Day Values', 'NWM_SWE_meters')

Visualize both the temporal and spatial variability of modeled SWE across the domain. 
To achieve this, we first create a subset of the modeled data by selecting only the first time step of each month from the simulated SWE dataset. To improve processing efficiency, we chunk the dataset along the time dimension, allowing faster access to individual time slices. We then apply a time filter and compute each selected slice individually to avoid excessive memory use. Finally, we combine the computed slices back into a single dataset for analysis and visualization.

In [ ]:
# Check the current chunks
ds_clip_donpedro.chunks

In [ ]:
# Rechunck
ds_clip_donpedro = ds_clip_donpedro.chunk({'time': 1})
ds_clip_donpedro.chunks

**Note**: The following code may take several minutes to run.

In [ ]:
%%time 

# Apply time filter
mask = (ds_clip_donpedro.time.dt.is_month_start) & (ds_clip_donpedro.time.dt.hour == 0)

# Compute in smaller pieces
selected_times = ds_clip_donpedro.time[mask]
parts = []
for t in selected_times.values:
    part = ds_clip_donpedro.sel(time=t).compute()  # One slice at a time
    parts.append(part)

# Combine the pieces
ds_monthly_start = xr.concat(parts, dim='time')

In [ ]:
data_var='SNEQV'
nwm_utils.plot_grid_vector_monthly_data(ds_monthly_start, data_var, donpedro_gdf, gdf_in_bbox)

<div style="background-color:#f5f5f5; padding:10px; border-left: 5px solid #007acc;">
  <h4>🧠 Reflect</h4>
  <p>
    In the plot above, you can see that in both accumulation and melt periods (e.g., Feb-Jun), snow was present in areas not covered by the existing observation points.  
How would having more observation points distributed across these snow-covered areas affect the basin-average SWE from observations, as calculated and plotted previously (`combined_df_mean`)?  
How could this change influence the bias, Nash-Sutcliffe Efficiency (NSE), and other statistical comparisons between modeled and observed SWE?  
And, how might improving the capture of spatial variability in SWE affect forecasts under future climate conditions?
  </p>
</div>